# 02C — Sentence Sample and Label

**Pipeline stage:**

* load AI-positive block corpus from 02B
* split blocks into sentence-level units
* materialize sentence shards
* build a stratified sentence sample
* label sampled sentences with DeepSeek
* export labeled sentence dataset

**Output directory**

`output/02C_sentence_sample/`

* `sentences_sample.parquet`
* `sentence_labels.jsonl`
* `sentence_labels.parquet`
* `sentences/part_*.parquet`



In [1]:
!pip -q install pyarrow aiohttp tqdm spacy

## Environment Setup

In [2]:
import os
import re
import json
import math
import time
import asyncio
from dataclasses import dataclass
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from tqdm.auto import tqdm

import aiohttp
import spacy

from google.colab import drive
drive.mount("/content/drive")

try:
    from google.colab import userdata
except Exception:
    userdata = None

Mounted at /content/drive


In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"

IN_DIR_02B = f"{BASE_DIR}/output/02B_block_train"
OUT_DIR_02C = f"{BASE_DIR}/output/02C_sentence_sample"

AI_BLOCKS_PARQUET = f"{IN_DIR_02B}/ai_blocks.parquet"

SENTENCES_DIR = f"{OUT_DIR_02C}/sentences"
SENTENCE_SAMPLE_PARQUET = f"{OUT_DIR_02C}/sentences_sample.parquet"
SENTENCE_LABELS_JSONL = f"{OUT_DIR_02C}/sentence_labels.jsonl"
SENTENCE_LABELS_PARQUET = f"{OUT_DIR_02C}/sentence_labels.parquet"

os.makedirs(OUT_DIR_02C, exist_ok=True)
os.makedirs(SENTENCES_DIR, exist_ok=True)

assert os.path.exists(AI_BLOCKS_PARQUET), f"Missing: {AI_BLOCKS_PARQUET}"

## Configuration

In [4]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SPACY_MODEL_CANDIDATES = [
    "en_core_web_sm",
]

MIN_SENT_CHARS = 15
MAX_SENT_CHARS = 800
MAX_SENTS_PER_BLOCK = 80

SENT_WRITE_BATCH = 250_000

SENTENCE_SAMPLE_N = 12000
TOP_DOMAIN_K = 40

# sample composition
SHARE_RANDOM = 0.55
SHARE_SUSPECT = 0.45

# DeepSeek API
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_ENDPOINT = f"{DEEPSEEK_BASE_URL}/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"

# concurrency
WORKERS = 64
QUEUE_SIZE = 512
JSONL_BATCH = 100

# HTTP tuning
HTTP_LIMIT = 128
HTTP_PER_HOST = 64
HTTP_TIMEOUT = 90
MAX_RETRIES = 5

# labeling prompt truncation
LABEL_MAX_CHARS = 1000

## Utilities

In [5]:
_ws_re = re.compile(r"\s+")
_non_text_heavy_re = re.compile(r"^[\W_]+$")
_menu_re = re.compile(
    r"(?i)\b("
    r"sign up|subscribe|newsletter|privacy|cookie|terms|advertisement|advertiser|"
    r"share on|share this|follow us|read more|continue reading|"
    r"open menu|close menu|main menu|navigation|skip to main content|"
    r"all rights reserved|contact us|login|log in|register|"
    r"recommended|related stories|distribution channels|published|updated|"
    r"copyright|press release|business wire|pr newswire|menu|search"
    r")\b"
)

_urlish_re = re.compile(r"(https?://|www\.|\.com\b|\.org\b|\.net\b)")
_jsish_re = re.compile(r"(?i)(function\s*\(|window\.|document\.|onclick|IntersectionObserver|var\s+\w+\s*=)")
_social_re = re.compile(r"(?i)\b(facebook|twitter|linkedin|reddit|discord|youtube|instagram|whatsapp)\b")

def normalize_spaces(text):
    return _ws_re.sub(" ", str(text or "")).strip()

def extract_domain(url):
    if not isinstance(url, str):
        return "NA"
    try:
        host = urlparse(url).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        return host if host else "NA"
    except Exception:
        return "NA"

def sentence_len_bucket(n):
    if n < 40:
        return "short"
    if n < 120:
        return "medium"
    return "long"

def suspect_sentence(text):
    t = str(text or "")
    if len(t) < 25:
        return 1
    if _menu_re.search(t):
        return 1
    if _urlish_re.search(t):
        return 1
    if _jsish_re.search(t):
        return 1
    if _social_re.search(t) and len(t) < 120:
        return 1
    if _non_text_heavy_re.fullmatch(t.strip() or " "):
        return 1
    punct = len(re.findall(r"[.!?]", t))
    alpha = len(re.findall(r"[A-Za-z]", t))
    if punct == 0 and alpha < 20:
        return 1
    return 0

def allocate_targets(strata_df: pd.DataFrame, total_n: int):
    keys = list(zip(strata_df["domain_tier"], strata_df["suspect_flag"]))
    raw = (strata_df["w"] * total_n).to_numpy()
    base = np.floor(raw).astype(int)
    base = np.maximum(base, 1)

    while base.sum() > total_n:
        idx = np.argmin(strata_df["w"].to_numpy())
        if base[idx] > 1:
            base[idx] -= 1
        else:
            candidates = np.where(base > 1)[0]
            if len(candidates) == 0:
                break
            base[candidates[0]] -= 1

    while base.sum() < total_n:
        idx = np.argmax(raw - base)
        base[idx] += 1

    return {keys[i]: int(base[i]) for i in range(len(keys))}

@dataclass
class Reservoir:
    k: int
    n_seen: int
    items: list

def reservoir_update(res: Reservoir, item: dict):
    res.n_seen += 1
    if len(res.items) < res.k:
        res.items.append(item)
        return
    j = np.random.randint(0, res.n_seen)
    if j < res.k:
        res.items[j] = item

## Load spaCy Sentence Splitter

In [6]:
def ensure_spacy_model(model_candidates):
    import sys
    import subprocess
    import importlib
    import spacy as _spacy

    last_error = None
    for model_name in model_candidates:
        try:
            nlp_local = _spacy.load(model_name, disable=["ner", "tagger", "lemmatizer", "attribute_ruler"])
            return nlp_local, model_name
        except Exception as e:
            last_error = e
            try:
                subprocess.check_call([sys.executable, "-m", "spacy", "download", model_name])
                importlib.invalidate_caches()
                nlp_local = _spacy.load(model_name, disable=["ner", "tagger", "lemmatizer", "attribute_ruler"])
                return nlp_local, model_name
            except Exception as e2:
                last_error = e2

    raise RuntimeError(f"Failed to load spaCy sentence model. Last error: {last_error}")

nlp, loaded_spacy_model = ensure_spacy_model(SPACY_MODEL_CANDIDATES)

if "parser" not in nlp.pipe_names and "senter" not in nlp.pipe_names and "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

print("Loaded spaCy model:", loaded_spacy_model)
print("Pipeline:", nlp.pipe_names)

Loaded spaCy model: en_core_web_sm
Pipeline: ['tok2vec', 'parser']


## Build Sentence Shards from AI Blocks

In [11]:
if len(os.listdir(SENTENCES_DIR)) == 0:
    ai_blocks = pd.read_parquet(AI_BLOCKS_PARQUET)
    print("ai_blocks shape:", ai_blocks.shape)

    required_cols = [
        "doc_id",
        "block_id",
        "block_uid",
        "url",
        "date",
        "language",
        "title",
        "domain",
        "block_text",
        "block_len",
        "p_ai_block",
        "pred_ai_block",
    ]
    missing = [c for c in required_cols if c not in ai_blocks.columns]
    assert not missing, f"Missing required columns: {missing}"

    ai_blocks["block_text"] = ai_blocks["block_text"].fillna("").astype(str)
    ai_blocks["domain"] = ai_blocks["domain"].fillna("NA").astype(str)

    records = []
    part = 0

    texts = ai_blocks["block_text"].tolist()

    for i, doc in enumerate(tqdm(nlp.pipe(texts, batch_size=512), total=len(texts), desc="Sentence splitting")):
        row = ai_blocks.iloc[i]

        sent_list = []
        for sent_id, sent in enumerate(doc.sents):
            s = normalize_spaces(sent.text)
            if len(s) < MIN_SENT_CHARS:
                continue
            if len(s) > MAX_SENT_CHARS:
                s = s[:MAX_SENT_CHARS]
            sent_list.append((sent_id, s))

        if len(sent_list) > MAX_SENTS_PER_BLOCK:
            sent_list = sent_list[:MAX_SENTS_PER_BLOCK]

        for sent_id, sent_text in sent_list:
            sent_uid = f"{row['block_uid']}:{sent_id}"
            records.append(
                {
                    "doc_id": int(row["doc_id"]),
                    "block_id": int(row["block_id"]),
                    "block_uid": str(row["block_uid"]),
                    "sent_id": int(sent_id),
                    "sent_uid": sent_uid,
                    "url": row["url"],
                    "date": row["date"],
                    "language": row["language"],
                    "title": row["title"],
                    "domain": row["domain"],
                    "sentence_text": sent_text,
                    "sent_len": len(sent_text),
                    "sent_len_bucket": sentence_len_bucket(len(sent_text)),
                    "suspect_flag": suspect_sentence(sent_text),
                    "p_ai_block": float(row["p_ai_block"]),
                    "pred_ai_block": int(row["pred_ai_block"]),
                }
            )

        if len(records) >= SENT_WRITE_BATCH:
            path = f"{SENTENCES_DIR}/part_{part:04d}.parquet"
            pd.DataFrame(records).to_parquet(path, index=False)
            records = []
            part += 1

    if records:
        path = f"{SENTENCES_DIR}/part_{part:04d}.parquet"
        pd.DataFrame(records).to_parquet(path, index=False)

    print("sentence shards written:", SENTENCES_DIR)
else:
    print("sentence shards already exist:", SENTENCES_DIR)

ai_blocks shape: (1184289, 14)


Sentence splitting:   0%|          | 0/1184289 [00:00<?, ?it/s]

sentence shards written: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02C_sentence_sample/sentences


## Build Stratified Sentence Sample

In [12]:
sentence_files = sorted(
    [
        os.path.join(SENTENCES_DIR, f)
        for f in os.listdir(SENTENCES_DIR)
        if f.endswith(".parquet")
    ]
)
assert sentence_files, f"No sentence shard files found in: {SENTENCES_DIR}"

# domain tier
sent_meta_parts = []
for fpath in tqdm(sentence_files, desc="Reading sentence metadata"):
    part = pd.read_parquet(fpath, columns=["domain"])
    sent_meta_parts.append(part)

sent_meta = pd.concat(sent_meta_parts, ignore_index=True)
domain_counts = sent_meta["domain"].value_counts()
top_domains = set(domain_counts.head(TOP_DOMAIN_K).index.tolist())

def domain_tier(x):
    return x if x in top_domains else "other"

del sent_meta_parts, sent_meta

random_target_n = int(SENTENCE_SAMPLE_N * SHARE_RANDOM)
suspect_target_n = SENTENCE_SAMPLE_N - random_target_n

# build approximate strata weights from full sentence corpus
strata_rows = []
for fpath in tqdm(sentence_files, desc="Building sample strata"):
    part = pd.read_parquet(fpath, columns=["domain", "suspect_flag"])
    part["domain_tier"] = part["domain"].map(domain_tier)
    tmp = (
        part.groupby(["domain_tier", "suspect_flag"], dropna=False)
        .size()
        .reset_index(name="n_rows")
    )
    strata_rows.append(tmp)

strata = (
    pd.concat(strata_rows, ignore_index=True)
    .groupby(["domain_tier", "suspect_flag"], dropna=False, as_index=False)
    .agg(n_rows=("n_rows", "sum"))
)

strata["w"] = strata["n_rows"] / strata["n_rows"].sum()

targets_random = allocate_targets(strata, random_target_n)
targets_suspect = allocate_targets(strata, suspect_target_n)

res_random = {k: Reservoir(v, 0, []) for k, v in targets_random.items()}
res_suspect = {k: Reservoir(v, 0, []) for k, v in targets_suspect.items()}

dataset = ds.dataset(SENTENCES_DIR, format="parquet")
scanner = dataset.scanner(columns=[
    "doc_id",
    "block_id",
    "block_uid",
    "sent_id",
    "sent_uid",
    "url",
    "date",
    "language",
    "title",
    "domain",
    "sentence_text",
    "sent_len",
    "sent_len_bucket",
    "suspect_flag",
    "p_ai_block",
    "pred_ai_block",
])

for batch in tqdm(scanner.to_batches(), desc="Sampling sentences"):
    pdf = batch.to_pandas()
    pdf["domain_tier"] = pdf["domain"].map(domain_tier)

    for r in pdf.itertuples(index=False):
        key = (r.domain_tier, r.suspect_flag)
        item = r._asdict()

        if key in res_random:
            reservoir_update(res_random[key], item)

        if int(r.suspect_flag) == 1 and key in res_suspect:
            reservoir_update(res_suspect[key], item)

sample_rows = []
for d in (res_random, res_suspect):
    for res in d.values():
        sample_rows.extend(res.items)

sample_df = pd.DataFrame(sample_rows)
sample_df = sample_df.drop_duplicates("sent_uid").reset_index(drop=True)

if len(sample_df) > SENTENCE_SAMPLE_N:
    sample_df = sample_df.sample(n=SENTENCE_SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)

sample_df.to_parquet(SENTENCE_SAMPLE_PARQUET, index=False)
print("sample size:", len(sample_df))
print("sample saved:", SENTENCE_SAMPLE_PARQUET)

Reading sentence metadata:   0%|          | 0/27 [00:00<?, ?it/s]

Building sample strata:   0%|          | 0/27 [00:00<?, ?it/s]

Sampling sentences: 0it [00:00, ?it/s]

sample size: 7302
sample saved: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02C_sentence_sample/sentences_sample.parquet


## DeepSeek Labeling

In [14]:
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "").strip()

if not DEEPSEEK_API_KEY and userdata:
    try:
        DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY").strip()
    except Exception:
        DEEPSEEK_API_KEY = ""

assert DEEPSEEK_API_KEY, "Missing DEEPSEEK_API_KEY."

SYSTEM_PROMPT = """
Return strict JSON.

Task:
Given one sentence extracted from an AI-related webpage block, determine whether the sentence is meaningful analytical content that should be retained for downstream NLP analysis.

Output schema:
{
  "is_content_sentence": 1,
  "confidence": 0.92,
  "reason_short": "brief explanation"
}

Label rules:
- is_content_sentence = 1 if the sentence contains meaningful article content, argument, fact, claim, description, quote, or analytically useful context.
- is_content_sentence = 0 if the sentence is webpage residue, template text, navigation, share rail, timestamp-only text, author/contact rail, subscribe/login prompt, copyright/footer text, category/tag rail, script residue, promo fragment, or other boilerplate.
- Keep the decision focused on sentence quality, not on whether the sentence is positive or negative.
- JSON only.
""".strip()

def user_prompt(row):
    sent = str(row.sentence_text or "")[:LABEL_MAX_CHARS]
    return f"""
URL: {row.url}
Domain: {row.domain}
Date: {row.date}
Title: {row.title}
Block AI score: {row.p_ai_block:.4f}
Suspect flag: {int(row.suspect_flag)}

Sentence:
{sent}
"""

def load_done_ids(path):
    if not os.path.exists(path):
        return set()
    done = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done.add(obj["sent_uid"])
            except Exception:
                continue
    return done

async def writer(queue, pbar):
    batch = []
    with open(SENTENCE_LABELS_JSONL, "a", encoding="utf-8") as f:
        while True:
            item = await queue.get()
            if item is None:
                break

            batch.append(item)
            pbar.update(1)

            if len(batch) >= JSONL_BATCH:
                for r in batch:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
                f.flush()
                batch = []

        if batch:
            for r in batch:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
            f.flush()

async def call_deepseek(session, headers, payload):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with session.post(
                DEEPSEEK_ENDPOINT,
                headers=headers,
                json=payload,
            ) as resp:
                txt = await resp.text()

                if resp.status in (429, 500, 502, 503, 504):
                    await asyncio.sleep(min(2 ** attempt, 20))
                    continue

                if resp.status != 200:
                    return {"error": f"http_{resp.status}", "error_raw": txt[:300]}

                data = json.loads(txt)
                content = data["choices"][0]["message"]["content"]
                parsed = json.loads(content)

                return {
                    "label_status": "ok",
                    "is_content_sentence": int(parsed.get("is_content_sentence", 0)),
                    "confidence": float(parsed.get("confidence", 0)),
                    "reason_short": str(parsed.get("reason_short", ""))[:200],
                }

        except Exception as e:
            if attempt == MAX_RETRIES:
                return {"label_status": "error", "error": str(e)[:300]}
            await asyncio.sleep(min(2 ** attempt, 20))

    return {"label_status": "error", "error": "max_retries"}

async def worker(session, headers, row_q, write_q):
    while True:
        row = await row_q.get()
        if row is None:
            break

        payload = {
            "model": DEEPSEEK_MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(row)},
            ],
            "temperature": 0,
            "max_tokens": 180,
            "response_format": {"type": "json_object"},
        }

        result = await call_deepseek(session, headers, payload)

        out = {
            "sent_uid": row.sent_uid,
            "block_uid": row.block_uid,
            "doc_id": int(row.doc_id),
            "block_id": int(row.block_id),
            "sent_id": int(row.sent_id),
            "sentence_text": str(row.sentence_text),
            "domain": str(row.domain),
            "title": str(row.title),
            "p_ai_block": float(row.p_ai_block),
            "suspect_flag": int(row.suspect_flag),
            **result,
        }

        await write_q.put(out)

async def run_labeling(sample_df):
    done = load_done_ids(SENTENCE_LABELS_JSONL)
    todo = sample_df[~sample_df["sent_uid"].isin(done)].copy()

    print("already labeled:", len(done))
    print("remaining:", len(todo))

    if len(todo) == 0:
        return

    row_q = asyncio.Queue(QUEUE_SIZE)
    write_q = asyncio.Queue()

    connector = aiohttp.TCPConnector(
        limit=HTTP_LIMIT,
        limit_per_host=HTTP_PER_HOST,
        ttl_dns_cache=300,
    )
    timeout = aiohttp.ClientTimeout(total=HTTP_TIMEOUT)

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    pbar = tqdm(total=len(todo), desc="Sentence labeling", dynamic_ncols=True)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        writer_task = asyncio.create_task(writer(write_q, pbar))
        workers = [
            asyncio.create_task(worker(session, headers, row_q, write_q))
            for _ in range(WORKERS)
        ]

        for row in todo.itertuples(index=False):
            await row_q.put(row)

        for _ in workers:
            await row_q.put(None)

        await asyncio.gather(*workers)
        await write_q.put(None)
        await writer_task

    pbar.close()

In [15]:
sample_df = pd.read_parquet(SENTENCE_SAMPLE_PARQUET)
await run_labeling(sample_df)

already labeled: 0
remaining: 7302


Sentence labeling:   0%|          | 0/7302 [00:00<?, ?it/s]

## Export Labeled Sentence Dataset

In [16]:
rows = []
with open(SENTENCE_LABELS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        try:
            rows.append(json.loads(line))
        except Exception:
            pass

labels = pd.DataFrame(rows)
sample_df = pd.read_parquet(SENTENCE_SAMPLE_PARQUET)

final = sample_df.merge(
    labels,
    on=["sent_uid", "block_uid", "doc_id", "block_id", "sent_id", "sentence_text", "domain", "title", "p_ai_block", "suspect_flag"],
    how="left",
)

final.to_parquet(SENTENCE_LABELS_PARQUET, index=False)

print("labeled sentence dataset written:", SENTENCE_LABELS_PARQUET)

labeled sentence dataset written: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02C_sentence_sample/sentence_labels.parquet


## Quick Readout

In [17]:
lab = pd.read_parquet(SENTENCE_LABELS_PARQUET)
print("raw labeled sentence shape:", lab.shape)

if "label_status" in lab.columns:
    print("\nlabel_status distribution:")
    print(lab["label_status"].value_counts(dropna=False))

if "is_content_sentence" in lab.columns:
    ok = lab[lab["label_status"] == "ok"].copy()
    print("\nusable labeled sentence rows:", len(ok))
    print("\ncontent label distribution:")
    print(ok["is_content_sentence"].value_counts(dropna=False))

    print("\nexample labeled rows:")
    display(
        ok[
            [
                "sent_uid",
                "domain",
                "suspect_flag",
                "is_content_sentence",
                "confidence",
                "sentence_text",
            ]
        ].head(20)
    )

raw labeled sentence shape: (7302, 21)

label_status distribution:
label_status
ok    7302
Name: count, dtype: int64

usable labeled sentence rows: 7302

content label distribution:
is_content_sentence
1    4729
0    2573
Name: count, dtype: int64

example labeled rows:


,sent_uid,domain,suspect_flag,is_content_sentence,confidence,sentence_text
0,73609:0:18,benzinga.com,0,1,0.85,"Human emotions can also drive decisions, influ..."
1,180393:0:13,benzinga.com,0,0,0.99,Cryptocurrency ScannersBest Business Crypto Ac...
2,197047:0:22,benzinga.com,0,0,0.98,Loading...Loading...Read Next:
3,89560:3:4,benzinga.com,0,1,0.95,"In the short term, this is good for companies ..."
4,72363:0:14,benzinga.com,0,0,0.99,ConferenceNewsEarningsInterviewsDealsRegulatio...
5,150322:0:9,benzinga.com,0,1,0.98,"""Basically… I’m amazed – it drives really, rea..."
6,52605:0:16,benzinga.com,0,1,0.95,The solutions demonstrate how Think Silicon is...
7,151291:2:56,benzinga.com,0,1,0.85,ServicesFor more information about this report...
8,23941:0:12,benzinga.com,0,1,0.95,"""It's almost at the beginning,"" Dimon noted, c..."
9,14786:1:11,benzinga.com,0,0,0.95,This story was generated using Benzinga Neuro ...
